# dist-send-recv-pair — faded example 3: Compute ring-topology left and right neighbor ranks with modular arithmetic

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`. Running the beacon reports progress on the `Distributed: dist.send/recv pair` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In a ring topology, each rank has a right neighbor `(rank + 1) % world_size` and a left neighbor `(rank - 1) % world_size`. The modular arithmetic wraps around so rank 0's left neighbor is rank `world_size - 1`, forming a complete ring. These neighbor ranks are used as `dst` in `dist.send` and `src` in `dist.recv` for one-step ring passes.

## Faded exercise 3

Complete `ring_exchange_step`. Compute the left and right neighbors, then send the local tensor to the right neighbor and receive the incoming tensor from the left neighbor.

**Fill in:** The two neighbor computations: `right = (rank + 1) % world_size` and `left = (rank - 1) % world_size`, followed by `dist_module.send(send_buf, dst=right)` and `dist_module.recv(recv_buf, src=left)`.

In [ ]:
def ring_exchange_step(rank, world_size, dist_module, send_buf, recv_buf):
    """One step of a ring pass: send right, receive from left."""
    right = (rank + 1) % world_size
    left  = (rank - 1) % world_size
    dist_module.send(send_buf, dst=right)
    dist_module.recv(recv_buf, src=left)


def _test():
    import torch
    from unittest.mock import MagicMock, call

    for world_size in [2, 4, 5]:
        for rank in range(world_size):
            dist_mock = MagicMock()
            send_buf = torch.zeros(3)
            recv_buf = torch.zeros(3)
            ring_exchange_step(rank, world_size, dist_mock, send_buf, recv_buf)

            expected_right = (rank + 1) % world_size
            expected_left  = (rank - 1) % world_size

            dist_mock.send.assert_called_once_with(send_buf, dst=expected_right)
            dist_mock.recv.assert_called_once_with(recv_buf, src=expected_left)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def ring_exchange_step(rank, world_size, dist_module, send_buf, recv_buf):
    """One step of a ring pass: send right, receive from left."""
    right = (rank + 1) % world_size
    left  = (rank - 1) % world_size
    dist_module.send(send_buf, dst=right)
    dist_module.recv(recv_buf, src=left)
```
</details>